# Lección 13 - Memoria del Agente


## Configuración

Este cuaderno muestra cómo crear un agente de reserva de viajes con **memoria persistente** usando el **Microsoft Agent Framework** (MAF).

Aprenderá cómo diferentes tipos de memoria del agente — de trabajo, a corto plazo y a largo plazo — afectan la forma en que un agente retiene y usa la información a lo largo de las conversaciones.

**Requisitos previos:**
- Un proyecto de Azure AI Foundry con un modelo de chat implementado (por ejemplo, `gpt-4o-mini`).
- Haber iniciado sesión con Azure CLI — ejecute `az login` en su terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — el punto de conexión de su proyecto Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — el nombre de su modelo implementado.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Tipos de memoria del agente

Los agentes de IA pueden aprovechar diferentes tipos de memoria, cada una con un propósito distinto:

### Memoria de trabajo
El propio hilo de conversación: los mensajes intercambiados en una sola sesión. El agente puede referirse a mensajes anteriores en el mismo hilo para mantener la coherencia. En MAF esto se crea mediante **`agent.create_session()`**, que devuelve un `AgentSession`.

### Memoria a corto plazo
Información que persiste durante la duración de una tarea o sesión pero no se almacena de forma permanente. Por ejemplo, el agente puede acumular hechos durante una conversación de planificación de varios turnos y usarlos para producir un itinerario final.

### Memoria a largo plazo
Preferencias y hechos que persisten **entre sesiones**. Un usuario que regresa no debería tener que repetir sus restricciones dietéticas o estilo de viaje. La memoria a largo plazo generalmente está respaldada por un almacenamiento externo: una base de datos, archivo o índice vectorial, y se presenta al agente mediante herramientas.


## Memoria de Trabajo con Sesiones

La forma más simple de memoria es la sesión de conversación. Cuando pasas la misma sesión (creada mediante `agent.create_session()`) a llamadas sucesivas de `agent.run()`, el agente ve todo el historial de esa conversación y puede recordar detalles anteriores.

Vamos a crear un agente de viajes y demostrar la memoria de trabajo.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

El agente recordó correctamente el presupuesto porque ambos mensajes comparten la misma sesión. Esto es **memoria de trabajo** — solo existe durante la vida útil de la sesión.

### ¿Qué sucede con un nuevo hilo?

Si creamos una sesión **nueva**, el agente no tiene memoria de la conversación anterior:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Patrón de Memoria a Largo Plazo

Para recordar las preferencias del usuario **a lo largo de las sesiones**, necesitamos un almacenamiento persistente que viva fuera del hilo de la conversación. El agente accede a este almacenamiento a través de **herramientas** — funciones que puede llamar para guardar y recuperar información.

A continuación implementamos un simple almacenamiento de preferencias en memoria (en producción lo respaldarías con una base de datos o un índice vectorial) y lo exponemos como herramientas que el agente puede usar.

### Arquitectura
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Escenario 1 — Usuario primerizo reserva un viaje de aniversario

Sarah visita por primera vez. El agente debe almacenar sus preferencias a través de las herramientas y usarlas para recomendar hoteles.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Escenario 2 — Sarah regresa semanas después

Sarah inicia un **nuevo hilo** (simulando una nueva sesión). La memoria de trabajo está vacía, pero el almacén de preferencias a largo plazo aún tiene su información. El agente debe recuperarla y usarla para personalizar las recomendaciones.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Resumen

En esta lección aprendiste tres tipos de memoria de agente y cómo implementarlas con el Microsoft Agent Framework:

| Tipo de Memoria | Mecanismo MAF | Duración |
|---|---|---|
| **De trabajo** | `agent.create_session()` | Conversación única |
| **A corto plazo** | Contexto acumulado dentro de un hilo | Tarea / sesión única |
| **A largo plazo** | Almacenamiento externo accesado vía funciones `@tool` | Entre sesiones |

### Puntos clave
1. **`agent.create_session()`** proporciona memoria de trabajo: el agente ve todo el historial de la conversación dentro de una sesión.
2. **Nuevas sesiones pierden contexto:** sin memoria a largo plazo, el agente no puede recordar conversaciones pasadas.
3. Las funciones **`@tool`** salvan la brecha: permiten que el agente guarde y recupere información de un almacenamiento persistente.
4. **La personalización mejora con el tiempo:** mientras más preferencias se almacenan, mejores son las recomendaciones del agente.

### Aplicaciones en el mundo real
- **Atención al cliente:** recordar el historial y preferencias del cliente
- **Asistentes personales:** mantener el contexto a lo largo de días o semanas
- **Salud:** rastrear información y preferencias del paciente
- **Comercio electrónico:** compras personalizadas basadas en historial

### Próximos pasos
- Reemplazar el diccionario en memoria con una base de datos o vector store (p. ej., Azure AI Search)
- Añadir expiración de memoria para información sensible al tiempo
- Construir sistemas multi-agente con memoria compartida
- Explorar el cuaderno Cognee para memoria respaldada por grafo de conocimiento


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso legal**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automáticas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda la traducción profesional por humanos. No nos hacemos responsables de malentendidos o interpretaciones erróneas derivadas del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
